In [3]:
import pandas as pd
#Reading the tools inference:
DDBB_Gender_w = pd.read_csv('CNKI_uniqueChineseGivenNames_gender_w.csv')
DDBB_Gender_r = pd.read_csv('CNKI_uniqueChineseGivenNames_gender_r.csv')
Threshold = 0
DDBB_Gender_r['gender'] = DDBB_Gender_r.NG.apply(lambda x: 'female' if x<Threshold else 
                                                     'male' if x>Threshold else 'neutral')

PD_names_gender = pd.read_csv('PD_names_gender_Namsor.csv')

Dict_Gender_w = {DDBB_Gender_w.givenname[i]:DDBB_Gender_w.gender[i] for i in DDBB_Gender_w.index}
Dict_Gender_r = {DDBB_Gender_r.givenname[i]:DDBB_Gender_r.gender[i] for i in DDBB_Gender_r.index}
Dict_Gender_n = {PD_names_gender.nameSearched[i]:PD_names_gender.likelyGender[i] for i in PD_names_gender.index}

#Creating a unique database:
DDBB_final = pd.DataFrame({'givenname': Dict_Gender_w.keys(), 
             'Gender_w': [Dict_Gender_w[k] for k in Dict_Gender_w.keys()],
             'Gender_r': [Dict_Gender_r[k] for k in Dict_Gender_r.keys()],
             'Gender_n': [Dict_Gender_n[k] for k in Dict_Gender_n.keys()]})

DDBB_final['Same_gender_all'] = [DDBB_final.Gender_w[i] == DDBB_final.Gender_r[i] == DDBB_final.Gender_n[i] for i in DDBB_final.index]
DDBB_final['Same_gender_n_r'] = [(DDBB_final.Gender_n[i] == DDBB_final.Gender_r[i]) & (DDBB_final.Gender_n[i] != DDBB_final.Gender_w[i]) for i in DDBB_final.index ]
DDBB_final['Same_gender_n_w'] = [(DDBB_final.Gender_n[i] == DDBB_final.Gender_w[i]) & (DDBB_final.Gender_n[i] != DDBB_final.Gender_r[i]) for i in DDBB_final.index ]
DDBB_final['Same_gender_w_r'] = [(DDBB_final.Gender_w[i] == DDBB_final.Gender_r[i]) & (DDBB_final.Gender_r[i] != DDBB_final.Gender_w[i]) for i in DDBB_final.index ]


def most_common(lst):
    return max(set(lst), key=lst.count)

List_gender_majority =  [most_common(list(DDBB_final[['Gender_w', 'Gender_r','Gender_n']].iloc[x]))for x in DDBB_final.index]

DDBB_final['Most_frequent_gender'] = List_gender_majority

# DDBB_final = pd.read_csv('DDBB_final_new.csv', delimiter='|')


#Reading the files from the surveys:
DDBD_test_remaining = pd.read_csv('Remaining_names_first_survey.csv', index_col=None)
Name_test_multiple_character= pd.read_csv('Names_test_multiple_character.csv', index_col=None)
Name_test_multiple_character_2 = pd.read_csv('Names_test_multiple_character_2.csv', index_col=None)


In [4]:
List_survey = list(DDBD_test_remaining.givenname)+list(Name_test_multiple_character.givenname)+list(Name_test_multiple_character_2.givenname)
DDBB_final[DDBB_final.givenname.isin(List_survey)][['Same_gender_all', 'Most_frequent_gender']].value_counts()


Same_gender_all  Most_frequent_gender
False            female                  30
                 male                    30
True             female                  10
                 male                    10
Name: count, dtype: int64